# Loss Weight Ablation Study

**Project A3010-251 — Autoencoder for Face Recognition**

This notebook runs ae_final with different RECON_WEIGHT / CLF_WEIGHT combinations
to justify the joint cost design. All other hyperparameters are identical.

| Run | RECON_WEIGHT | CLF_WEIGHT | Description |
|-----|-------------|------------|-------------|
| A | 1.0 | 0.0 | Pure reconstruction |
| B | 1.0 | 0.5 | **Current best** (skip — already have result) |
| C | 1.0 | 1.0 | Equal weighting |
| D | 0.0 | 1.0 | Pure classification |

In [ ]:
import os, glob, json, time, random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from collections import Counter

print(f'TF {tf.__version__}')

In [ ]:
# ══════════════════════════════════════════════
# CONFIGURATION (identical to ae_final except weights)
# ══════════════════════════════════════════════

DATA_DIR = '/kaggle/input/datasets/jessicali9530/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled'
OUT_DIR  = '/kaggle/working/ablation_results'
os.makedirs(OUT_DIR, exist_ok=True)

IMG_SIZE          = 128
BATCH_SIZE        = 64
SEED              = 42
MIN_IMAGES_PER_ID = 10
TOP_K             = 200
TRAIN_RATIO       = 0.70
VAL_RATIO         = 0.15

LATENT_DIM          = 512
AE_EPOCHS           = 80
CLF_EPOCHS_FROZEN   = 40
CLF_EPOCHS_FINETUNE = 60
AE_LR               = 1e-3
CLF_LR_FROZEN       = 3e-4
CLF_LR_FINETUNE     = 5e-5

# ── Weight combinations to test ──
# Skip (1.0, 0.5) since we already have that result: 73.47% / 94.12%
WEIGHT_COMBOS = [
    (1.0, 0.0, 'Pure Reconstruction'),
    (1.0, 1.0, 'Equal Weighting'),
    (0.0, 1.0, 'Pure Classification'),
]

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('Config OK')

In [ ]:
# ══════════════════════════════════════════════
# DATA LOADING (runs once, reused for all combos)
# ══════════════════════════════════════════════

# Identity filtering
people = []
for person in os.listdir(DATA_DIR):
    pdir = os.path.join(DATA_DIR, person)
    if not os.path.isdir(pdir): continue
    imgs = glob.glob(os.path.join(pdir, '*.jpg')) + glob.glob(os.path.join(pdir, '*.png'))
    if len(imgs) >= MIN_IMAGES_PER_ID:
        people.append((person, len(imgs)))

people.sort(key=lambda x: x[1], reverse=True)
people = people[:TOP_K]

class_names = [p for p, _ in people]
class_to_idx = {c: i for i, c in enumerate(class_names)}
NUM_CLASSES = len(class_names)

# 70/15/15 split
random.seed(SEED)
train_paths, train_labels = [], []
val_paths, val_labels = [], []
test_paths, test_labels = [], []

for person, _ in people:
    pdir = os.path.join(DATA_DIR, person)
    imgs = sorted(glob.glob(os.path.join(pdir, '*.jpg')) + glob.glob(os.path.join(pdir, '*.png')))
    random.shuffle(imgs)
    n = len(imgs)
    n_train = max(1, int(TRAIN_RATIO * n))
    n_val = max(1, int(VAL_RATIO * n))
    if n_train + n_val >= n:
        n_train = max(1, n - n_val - 1)
    n_test = n - n_train - n_val
    if n_test < 1:
        n_train -= 1; n_test = 1
    label = class_to_idx[person]
    train_paths.extend(imgs[:n_train]); train_labels.extend([label]*n_train)
    val_paths.extend(imgs[n_train:n_train+n_val]); val_labels.extend([label]*n_val)
    test_paths.extend(imgs[n_train+n_val:]); test_labels.extend([label]*n_test)

# ALL LFW for Phase 1
random.seed(SEED)
all_people_dirs = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
MAX_PER_PERSON = 20
ae_all_paths, ae_all_labels = [], []
for person in all_people_dirs:
    pdir = os.path.join(DATA_DIR, person)
    imgs = sorted(glob.glob(os.path.join(pdir, '*.jpg')) + glob.glob(os.path.join(pdir, '*.png')))
    if len(imgs) > 0:
        random.shuffle(imgs)
        selected = imgs[:MAX_PER_PERSON]
        ae_all_paths.extend(selected)
        ae_all_labels.extend([class_to_idx.get(person, -1)] * len(selected))

combined = list(zip(ae_all_paths, ae_all_labels))
random.shuffle(combined)
ae_all_paths, ae_all_labels = zip(*combined)
ae_all_paths, ae_all_labels = list(ae_all_paths), list(ae_all_labels)
ae_split = int(0.8 * len(ae_all_paths))
ae_train_paths, ae_train_labels = ae_all_paths[:ae_split], ae_all_labels[:ae_split]
ae_val_paths, ae_val_labels = ae_all_paths[ae_split:], ae_all_labels[ae_split:]

print(f'Classes: {NUM_CLASSES}')
print(f'Train: {len(train_paths)}  Val: {len(val_paths)}  Test: {len(test_paths)}')
print(f'AE train: {len(ae_train_paths)}  AE val: {len(ae_val_paths)}')

In [ ]:
# ══════════════════════════════════════════════
# HELPER FUNCTIONS (reused for all combos)
# ══════════════════════════════════════════════
AUTOTUNE = tf.data.AUTOTUNE

def load_image(fp):
    img = tf.io.read_file(fp)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(img, tf.float32) / 255.0

def augment(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.image.random_saturation(img, 0.85, 1.15)
    cs = tf.random.uniform([], minval=int(IMG_SIZE*0.85), maxval=IMG_SIZE+1, dtype=tf.int32)
    img = tf.image.random_crop(img, [cs, cs, 3])
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.clip_by_value(img, 0.0, 1.0)

def load_ae_train(fp, label): return augment(load_image(fp)), label
def load_ae_val(fp, label): return load_image(fp), label
def load_clf_train(fp, y): return augment(load_image(fp)), y
def load_clf_val(fp, y): return load_image(fp), y

def make_datasets():
    train_ae = tf.data.Dataset.from_tensor_slices((ae_train_paths, ae_train_labels)).shuffle(len(ae_train_paths), seed=SEED).map(load_ae_train, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
    val_ae = tf.data.Dataset.from_tensor_slices((ae_val_paths, ae_val_labels)).map(load_ae_val, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
    train_clf = tf.data.Dataset.from_tensor_slices((train_paths, train_labels)).shuffle(2000, seed=SEED).map(load_clf_train, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
    val_clf = tf.data.Dataset.from_tensor_slices((val_paths, val_labels)).map(load_clf_val, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
    test_clf = tf.data.Dataset.from_tensor_slices((test_paths, test_labels)).map(load_clf_val, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return train_ae, val_ae, train_clf, val_clf, test_clf

def ssim_mse_loss(y_true, y_pred, alpha=0.7):
    ssim_val = tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))
    mse_val = tf.reduce_mean(tf.square(y_true - y_pred))
    return alpha * (1.0 - ssim_val) + (1.0 - alpha) * mse_val

print('Helpers ready')

In [ ]:
# ══════════════════════════════════════════════
# MODEL BUILDING FUNCTION
# ══════════════════════════════════════════════

def enc_block(x, filters, n_convs, block_name, use_batch_norm=True):
    for i in range(n_convs):
        x = layers.Conv2D(filters, (3,3), padding='same', kernel_initializer='he_normal', name=f'{block_name}_conv{i+1}')(x)
        if use_batch_norm:
            x = layers.BatchNormalization(name=f'{block_name}_bn{i+1}')(x)
        x = layers.Activation('relu', name=f'{block_name}_relu{i+1}')(x)
    x = layers.MaxPooling2D((2,2), strides=2, name=f'{block_name}_pool')(x)
    return x

def dec_block(x, filters, n_convs, block_name, use_batch_norm=True, final_block=False):
    x = layers.UpSampling2D((2,2), name=f'{block_name}_upsample')(x)
    for i in range(n_convs):
        is_last = final_block and (i == n_convs - 1)
        x = layers.Conv2D(filters if not is_last else 3, (3,3), padding='same', kernel_initializer='he_normal', name=f'{block_name}_conv{i+1}')(x)
        if is_last:
            x = layers.Activation('sigmoid', name='dec_sigmoid')(x)
        else:
            if use_batch_norm:
                x = layers.BatchNormalization(name=f'{block_name}_bn{i+1}')(x)
            x = layers.Activation('relu', name=f'{block_name}_relu{i+1}')(x)
    return x

def build_models(num_classes):
    inp = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='encoder_input')
    x = enc_block(inp, 64, 2, 'enc_b1')
    x = enc_block(x, 128, 2, 'enc_b2')
    x = enc_block(x, 256, 3, 'enc_b3')
    x = enc_block(x, 512, 3, 'enc_b4')
    x = enc_block(x, 512, 3, 'enc_b5')
    x = layers.GlobalAveragePooling2D(name='enc_gap')(x)
    z = layers.Dense(LATENT_DIM, name='latent_embedding')(x)
    z = layers.BatchNormalization(name='latent_bn')(z)
    z = layers.Activation('relu', name='latent_relu')(z)
    encoder = Model(inp, z, name='Encoder')
    
    x_dec = layers.Dense(4*4*512, name='dec_dense')(z)
    x_dec = layers.Reshape((4,4,512), name='dec_reshape')(x_dec)
    x_dec = dec_block(x_dec, 512, 3, 'dec_b5')
    x_dec = dec_block(x_dec, 256, 3, 'dec_b4')
    x_dec = dec_block(x_dec, 128, 3, 'dec_b3')
    x_dec = dec_block(x_dec, 64, 2, 'dec_b2')
    x_dec = dec_block(x_dec, 32, 2, 'dec_b1', final_block=True)
    autoencoder = Model(inp, x_dec, name='Face_Autoencoder')
    
    x_clf = layers.Dense(512, kernel_initializer='he_normal', name='joint_clf_fc1')(z)
    x_clf = layers.BatchNormalization(name='joint_clf_bn1')(x_clf)
    x_clf = layers.Activation('relu', name='joint_clf_relu1')(x_clf)
    x_clf = layers.Dropout(0.5, name='joint_clf_drop1')(x_clf)
    x_clf = layers.Dense(256, kernel_initializer='he_normal', name='joint_clf_fc2')(x_clf)
    x_clf = layers.BatchNormalization(name='joint_clf_bn2')(x_clf)
    x_clf = layers.Activation('relu', name='joint_clf_relu2')(x_clf)
    x_clf = layers.Dropout(0.3, name='joint_clf_drop2')(x_clf)
    clf_out = layers.Dense(num_classes, activation='softmax', name='joint_clf_softmax')(x_clf)
    
    joint_model = Model(inp, [x_dec, clf_out], name='Joint_AE_Classifier')
    return encoder, autoencoder, joint_model

print('Model builder ready')

In [ ]:
# ══════════════════════════════════════════════
# FULL TRAINING + EVAL PIPELINE FOR ONE COMBO
# ══════════════════════════════════════════════

def run_experiment(recon_w, clf_w, label, num_classes):
    print(f'\n{"="*60}')
    print(f'  RUNNING: {label}')
    print(f'  RECON_WEIGHT={recon_w}  CLF_WEIGHT={clf_w}')
    print(f'{"="*60}\n')
    
    # Fresh model
    tf.keras.backend.clear_session()
    random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
    encoder, autoencoder, joint_model = build_models(num_classes)
    
    # Datasets
    train_ae, val_ae, train_clf, val_clf, test_clf = make_datasets()
    
    # ── Phase 1: Joint training ──
    optimizer_p1 = keras.optimizers.Adam(learning_rate=AE_LR)
    clf_loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=False)
    
    RW = tf.constant(recon_w, dtype=tf.float32)
    CW = tf.constant(clf_w, dtype=tf.float32)
    
    @tf.function
    def train_step(images, labels):
        with tf.GradientTape() as tape:
            recon, clf_probs = joint_model(images, training=True)
            recon_loss = ssim_mse_loss(images, recon)
            known_mask = tf.cast(labels >= 0, tf.float32)
            safe_labels = tf.maximum(labels, 0)
            per_sample = keras.losses.sparse_categorical_crossentropy(safe_labels, clf_probs)
            n_known = tf.maximum(tf.reduce_sum(known_mask), 1.0)
            clf_loss = tf.reduce_sum(per_sample * known_mask) / n_known
            total_loss = RW * recon_loss + CW * clf_loss
        grads = tape.gradient(total_loss, joint_model.trainable_variables)
        optimizer_p1.apply_gradients(zip(grads, joint_model.trainable_variables))
        clf_preds = tf.argmax(clf_probs, axis=1)
        correct = tf.cast(clf_preds == tf.cast(safe_labels, tf.int64), tf.float32)
        clf_acc = tf.reduce_sum(correct * known_mask) / n_known
        return total_loss, recon_loss, clf_loss, clf_acc
    
    @tf.function
    def val_step(images, labels):
        recon, clf_probs = joint_model(images, training=False)
        recon_loss = ssim_mse_loss(images, recon)
        known_mask = tf.cast(labels >= 0, tf.float32)
        safe_labels = tf.maximum(labels, 0)
        per_sample = keras.losses.sparse_categorical_crossentropy(safe_labels, clf_probs)
        n_known = tf.maximum(tf.reduce_sum(known_mask), 1.0)
        clf_loss = tf.reduce_sum(per_sample * known_mask) / n_known
        total_loss = RW * recon_loss + CW * clf_loss
        clf_preds = tf.argmax(clf_probs, axis=1)
        correct = tf.cast(clf_preds == tf.cast(safe_labels, tf.int64), tf.float32)
        clf_acc = tf.reduce_sum(correct * known_mask) / n_known
        return total_loss, recon_loss, clf_loss, clf_acc
    
    best_val = float('inf')
    patience_ctr = 0
    current_lr = AE_LR
    lr_wait = 0
    prev_vl = float('inf')
    p1_start = time.time()
    
    for epoch in range(AE_EPOCHS):
        t_t, t_r, t_c, t_a = [], [], [], []
        for bi, bl in train_ae:
            tl, rl, cl, ca = train_step(bi, bl)
            t_t.append(float(tl)); t_r.append(float(rl)); t_c.append(float(cl)); t_a.append(float(ca))
        v_t, v_r, v_c, v_a = [], [], [], []
        for bi, bl in val_ae:
            tl, rl, cl, ca = val_step(bi, bl)
            v_t.append(float(tl)); v_r.append(float(rl)); v_c.append(float(cl)); v_a.append(float(ca))
        vl_total = np.mean(v_t)
        
        improved = ''
        if vl_total < best_val:
            best_val = vl_total
            joint_model.save_weights(os.path.join(OUT_DIR, f'p1_best_{label.replace(" ","_")}.weights.h5'))
            patience_ctr = 0; improved = ' << saved'
        else:
            patience_ctr += 1
        if vl_total >= prev_vl:
            lr_wait += 1
            if lr_wait >= 4:
                current_lr = max(current_lr * 0.5, 1e-6)
                optimizer_p1.learning_rate.assign(current_lr)
                lr_wait = 0
        else:
            lr_wait = 0
        prev_vl = vl_total
        
        if (epoch+1) % 10 == 0 or improved:
            print(f'  E{epoch+1:2d}/{AE_EPOCHS} loss:{np.mean(t_t):.4f} recon:{np.mean(t_r):.4f} clf:{np.mean(t_c):.4f} acc:{np.mean(t_a):.3f} | val:{vl_total:.4f}{improved}')
        if patience_ctr >= 10:
            print(f'  Early stop at epoch {epoch+1}')
            break
    
    joint_model.load_weights(os.path.join(OUT_DIR, f'p1_best_{label.replace(" ","_")}.weights.h5'))
    p1_time = time.time() - p1_start
    print(f'  Phase 1 done: {p1_time/60:.1f} min')
    
    # ── Phase 2a: Frozen encoder + classifier head ──
    for layer in encoder.layers:
        layer.trainable = False
    
    inp2 = encoder.input
    z2 = encoder.output
    x = layers.Dense(512, kernel_initializer='he_normal', name='clf_fc1')(z2)
    x = layers.BatchNormalization(name='clf_bn1')(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, kernel_initializer='he_normal', name='clf_fc2')(x)
    x = layers.BatchNormalization(name='clf_bn2')(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)
    out2 = layers.Dense(num_classes, activation='softmax', name='clf_softmax')(x)
    clf_model = Model(inp2, out2, name='AE_Classifier')
    
    clf_model.compile(optimizer=keras.optimizers.Adam(CLF_LR_FROZEN),
                      loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    h2a = clf_model.fit(train_clf, validation_data=val_clf, epochs=CLF_EPOCHS_FROZEN,
                        callbacks=[keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
                                   keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)],
                        verbose=0)
    print(f'  Phase 2a best val_acc: {max(h2a.history["val_accuracy"]):.4f}')
    
    # ── Phase 2b: Fine-tune all layers ──
    for layer in encoder.layers:
        layer.trainable = True
    
    def get_lr_mult(name):
        if 'enc_b1' in name or 'enc_b2' in name: return 0.01
        if 'enc_b3' in name: return 0.1
        if 'enc_b4' in name: return 0.5
        if 'enc_b5' in name or 'enc_gap' in name or 'latent' in name: return 1.0
        return 5.0  # classifier layers
    
    ft_optimizer = keras.optimizers.Adam(CLF_LR_FINETUNE)
    ce_loss = keras.losses.SparseCategoricalCrossentropy(from_logits=False)
    
    @tf.function
    def ft_train_step(images, labels):
        with tf.GradientTape() as tape:
            preds = clf_model(images, training=True)
            loss = ce_loss(labels, preds)
        grads = tape.gradient(loss, clf_model.trainable_variables)
        scaled = []
        for g, v in zip(grads, clf_model.trainable_variables):
            if g is not None:
                scaled.append(g * get_lr_mult(v.name))
            else:
                scaled.append(g)
        ft_optimizer.apply_gradients(zip(scaled, clf_model.trainable_variables))
        acc = tf.reduce_mean(tf.cast(tf.argmax(preds, axis=1) == tf.cast(labels, tf.int64), tf.float32))
        return loss, acc
    
    best_ft_acc = 0
    ft_patience = 0
    for epoch in range(CLF_EPOCHS_FINETUNE):
        losses, accs = [], []
        for bi, bl in train_clf:
            l, a = ft_train_step(bi, bl)
            losses.append(float(l)); accs.append(float(a))
        # Val
        v_accs = []
        for bi, bl in val_clf:
            p = clf_model(bi, training=False)
            va = tf.reduce_mean(tf.cast(tf.argmax(p, axis=1) == tf.cast(bl, tf.int64), tf.float32))
            v_accs.append(float(va))
        va_mean = np.mean(v_accs)
        if va_mean > best_ft_acc:
            best_ft_acc = va_mean
            clf_model.save_weights(os.path.join(OUT_DIR, f'ft_best_{label.replace(" ","_")}.weights.h5'))
            ft_patience = 0
        else:
            ft_patience += 1
        if (epoch+1) % 10 == 0:
            print(f'  FT E{epoch+1} loss:{np.mean(losses):.4f} acc:{np.mean(accs):.3f} val_acc:{va_mean:.3f}')
        if ft_patience >= 10:
            print(f'  FT early stop at epoch {epoch+1}')
            break
    
    clf_model.load_weights(os.path.join(OUT_DIR, f'ft_best_{label.replace(" ","_")}.weights.h5'))
    
    # ── Evaluate on test set ──
    all_preds = []
    all_labels_list = []
    for bi, bl in test_clf:
        p = clf_model(bi, training=False)
        all_preds.append(p.numpy())
        all_labels_list.append(bl.numpy())
    
    all_preds = np.concatenate(all_preds, axis=0)
    all_true = np.concatenate(all_labels_list, axis=0)
    
    top1_preds = np.argmax(all_preds, axis=1)
    top1_acc = np.mean(top1_preds == all_true)
    
    top5_preds = np.argsort(all_preds, axis=1)[:, -5:]
    top5_acc = np.mean([all_true[i] in top5_preds[i] for i in range(len(all_true))])
    
    total_time = time.time() - p1_start
    print(f'\n  ┌─────────────────────────────────────┐')
    print(f'  │ {label:>35s} │')
    print(f'  │ Top-1: {top1_acc*100:6.2f}%  Top-5: {top5_acc*100:6.2f}% │')
    print(f'  │ Time: {total_time/60:.1f} min                     │')
    print(f'  └─────────────────────────────────────┘\n')
    
    return {
        'label': label,
        'recon_weight': recon_w,
        'clf_weight': clf_w,
        'top1': round(top1_acc * 100, 2),
        'top5': round(top5_acc * 100, 2),
        'time_min': round(total_time / 60, 1),
    }

print('Pipeline ready')

In [ ]:
# ══════════════════════════════════════════════
# RUN ALL COMBINATIONS
# ══════════════════════════════════════════════

results = []

for rw, cw, label in WEIGHT_COMBOS:
    r = run_experiment(rw, cw, label, NUM_CLASSES)
    results.append(r)

# Add the known result for (1.0, 0.5)
results.append({
    'label': 'Joint (Current Best)',
    'recon_weight': 1.0,
    'clf_weight': 0.5,
    'top1': 73.47,
    'top5': 94.12,
    'time_min': 0,
})

# Sort by clf_weight for nice display
results.sort(key=lambda x: (x['recon_weight'], x['clf_weight']))

print('\nAll experiments complete!')

In [ ]:
# ══════════════════════════════════════════════
# RESULTS TABLE
# ══════════════════════════════════════════════

print('\n' + '='*70)
print('  LOSS WEIGHT ABLATION RESULTS')
print('='*70)
print(f'{"RECON_W":>10s} {"CLF_W":>8s} {"Top-1":>8s} {"Top-5":>8s} {"Time":>8s}  Description')
print('-'*70)
for r in results:
    marker = ' ◀ best' if r['top1'] == max(x['top1'] for x in results) else ''
    print(f'{r["recon_weight"]:>10.1f} {r["clf_weight"]:>8.1f} {r["top1"]:>7.2f}% {r["top5"]:>7.2f}% {r["time_min"]:>7.1f}m  {r["label"]}{marker}')
print('='*70)

# Save results
with open(os.path.join(OUT_DIR, 'ablation_results.json'), 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResults saved to {OUT_DIR}/ablation_results.json')